# Q3 baseline: the co-author seed and first journal entry

The question

- does having a co-author who already published in a journal raise the chance of entering that journal for the first time

The unit is one opportunity, a row per (author, journal, year) where the author was active and had not entered the journal yet, from the event table built in `logic/build_event_table.py`. `C` is the seed, an earlier collaborator who was in the journal before year t. `F` is first entry in year t. Q3 proper is the topic-adjusted ratio, so everything here splits in two

1. the naive contrast `F ~ C`, computable today and explicitly unadjusted
2. a provisional adjustment `F ~ C + T` with Pierre's rolling topic fit, marked provisional because three things about `T` are still open, the timing of the three-paper threshold, the anchor rule for the frozen variant, and a status column that explains why empty cells are empty

The canonical numbers come later, after those answers and after Lennart's Prolog table confirms the event table. Variant A is the main opportunity set and B the sensitivity, as agreed, so this notebook reads variant A only.

Inputs, both local in `../data/` and not in git

- `event_table_python_v0_oppA.csv`, 6,422,558 rows
- `event_table_topicmatch.csv`, the same rows with Pierre's `topic_match` added

## Step 1: load and align

- both files carry the same rows in the same order, which is asserted on the key columns before the topic column is attached
- `F` is read off `entering_work_id`, filled exactly on entry rows

In [1]:
import numpy as np, pandas as pd

usecols = ["author_id", "journal_id", "t", "n_prior_papers", "coauthor_seed",
           "first_entry_ride", "entering_work_id"]
ev = pd.read_csv("../data/event_table_python_v0_oppA.csv", usecols=usecols, low_memory=False,
                 dtype={"t": "int16", "n_prior_papers": "int32", "coauthor_seed": "int8",
                        "first_entry_ride": "int8", "entering_work_id": "str"})
tm = pd.read_csv("../data/event_table_topicmatch.csv", low_memory=False,
                 usecols=["author_id", "journal_id", "t", "topic_match", "tm_status", "profile_cutoff"],
                 dtype={"t": "int16"})

# same rows in the same order, otherwise attaching by position would silently mix rows
assert len(ev) == len(tm)
assert (ev["author_id"].values == tm["author_id"].values).all()
assert (ev["journal_id"].values == tm["journal_id"].values).all()
assert (ev["t"].values == tm["t"].values).all()
ev["T"] = tm["topic_match"].values
ev["tm_status"] = tm["tm_status"].values

# the status column must explain the fill exactly, and every profile must stop before t
assert ((tm["tm_status"] == "ok") == tm["topic_match"].notna()).all()
assert (tm.loc[tm.tm_status == "ok", "profile_cutoff"] < tm.loc[tm.tm_status == "ok", "t"]).all()
del tm

ev["F"] = ev["entering_work_id"].notna().astype("int8")
C = ev["coauthor_seed"].values
F = ev["F"].values
print(f"{len(ev):,} rows, {int(F.sum()):,} entries, {(C==1).sum():,} rows with a seed")

6,422,558 rows, 96,819 entries, 19,035 rows with a seed


## Step 2: check against the build

The event table build printed its own headline counts. Rebuilding them here from the loaded frame confirms nothing was lost or doubled on the way in.

In [2]:
assert int(F.sum()) == 96819                       # one entry row per (author, journal) pair in the corpus
assert int((C == 1).sum()) == 19035                # rows where the seed predates t
assert int(F[C == 1].sum()) == 1784                # entries that happened with a seed in place
assert int(ev["first_entry_ride"].sum()) == 756    # entries with the qualifying seed co-author on the paper
assert ev["T"].notna().sum() == 623510             # rows where Pierre's rolling topic fit exists
print("all five counts match the build")

all five counts match the build


## Step 3: the naive contrast

- entry rate with a seed against entry rate without, nothing held constant
- this number is confounded by design, an author with a seed is better connected and probably closer to the journal's topics, so it is a ceiling for the effect, not the effect
- the interval resamples whole authors, because one author contributes many related rows and row-level intervals would be too narrow

In [3]:
p1, p0 = F[C == 1].mean(), F[C == 0].mean()
print(f"entry rate with a seed    {p1:.4%}  ({int(F[C==1].sum()):,} of {(C==1).sum():,})")
print(f"entry rate without        {p0:.4%}  ({int(F[C==0].sum()):,} of {(C==0).sum():,})")
print(f"naive rate ratio          {p1/p0:.2f}")

# author level counts once, then the bootstrap only touches these four arrays
codes, authors = pd.factorize(ev["author_id"])
nA = len(authors)
n1 = np.bincount(codes[C == 1], minlength=nA)
e1 = np.bincount(codes[(C == 1) & (F == 1)], minlength=nA)
n0 = np.bincount(codes[C == 0], minlength=nA)
e0 = np.bincount(codes[(C == 0) & (F == 1)], minlength=nA)

rng = np.random.default_rng(31)
reps = []
for _ in range(2000):
    idx = rng.integers(0, nA, nA)   # draw authors with replacement, rows follow their author
    a, b, c, d = e1[idx].sum(), n1[idx].sum(), e0[idx].sum(), n0[idx].sum()
    if b and c and d:
        reps.append((a / b) / (c / d))
lo, hi = np.percentile(reps, [2.5, 97.5])
print(f"author-clustered 95% CI   [{lo:.2f}, {hi:.2f}]  ({len(reps)} bootstrap draws)")

entry rate with a seed    9.3722%  (1,784 of 19,035)
entry rate without        1.4841%  (95,035 of 6,403,523)
naive rate ratio          6.32


author-clustered 95% CI   [6.04, 6.59]  (2000 bootstrap draws)


## Step 4: ride against independent, inside the seeded entries

Of the entries that happened with a seed in place, some carry the seed co-author on the entering paper itself and some do not. The split matters because riding along and entering independently are different mechanisms, and only the seeded entries can show it.

In [4]:
rides = int(ev.loc[(C == 1) & (F == 1), "first_entry_ride"].sum())
seeded_entries = int(F[C == 1].sum())
print(f"seeded entries {seeded_entries:,}")
print(f"  with the seed co-author on the entering paper   {rides:,}  ({rides/seeded_entries:.0%})")
print(f"  without, so independent of that person          {seeded_entries-rides:,}  ({1-rides/seeded_entries:.0%})")

seeded entries 1,784
  with the seed co-author on the entering paper   756  (42%)
  without, so independent of that person          1,028  (58%)


## Step 5: stability by year

The corpus starts in 2015, so the first years cannot carry seeds, no co-author relation can predate the corpus. The seeded cells only reach a usable size around 2021, everything before rides on a handful of entries and should not be read as a trend.

In [5]:
g = ev.groupby("t", observed=True)
yr = pd.DataFrame({
    "seeded_rows": g.apply(lambda x: int((x.coauthor_seed == 1).sum()), include_groups=False),
    "seeded_entries": g.apply(lambda x: int(((x.coauthor_seed == 1) & (x.F == 1)).sum()), include_groups=False),
    "other_rows": g.apply(lambda x: int((x.coauthor_seed == 0).sum()), include_groups=False),
    "other_entries": g.apply(lambda x: int(((x.coauthor_seed == 0) & (x.F == 1)).sum()), include_groups=False),
})
yr["rate_ratio"] = (yr.seeded_entries / yr.seeded_rows) / (yr.other_entries / yr.other_rows)
print(yr.to_string(float_format=lambda v: f"{v:.2f}"))
print("\nread the ratio from 2021 on, the cells before are single digit seeded entries")

      seeded_rows  seeded_entries  other_rows  other_entries  rate_ratio
t                                                                       
2015            0               0      117056           1855         NaN
2016            9               5      161085           2461       36.36
2017           46              12      196118           2952       17.33
2018          185              39      326591           4906       14.03
2019          512              77      558551           8475        9.91
2020         1001             141      723699          10631        9.59
2021         1739             170      856449          12734        6.57
2022         3314             343     1052619          15723        6.93
2023         5243             441     1122706          16497        5.72
2024         6986             556     1288649          18801        5.46

read the ratio from 2021 on, the cells before are single digit seeded entries


## Step 6: what T covers, with the reasons now named

`topic_match` is Pierre's rolling author profile against the journal, 0 to 1, and the diagnostic export confirms the timing in the data itself, `profile_cutoff` sits strictly before t on every filled row. Every empty cell names its reason in `tm_status`, so this audit counts instead of inferring

- the three-paper threshold counts an author's whole 2015 to 2024 output with an abstract, so future productivity decides who gets a T at all, Pierre removes it in the next run
- journal profiles build only from the selected authors' papers, so a journal can lack a profile before t even for a well covered author
- `n_profile_papers` and `n_journal_papers` are written as zero on below-threshold rows in this run, so the real counts are missing exactly where they would explain the most, known and queued for the rerun

In [6]:
has = ev["T"].notna().values
print(f"T exists on {has.sum():,} rows ({has.mean():.1%})")
print(f"  on entry rows            {has[F==1].mean():.1%}")
print(f"  on seeded rows           {has[C==1].mean():.1%}")
print(f"  on seeded entries        {has[(C==1)&(F==1)].mean():.1%}")

print("\nwhy the rest is empty, all rows")
print(ev["tm_status"].value_counts().to_string())
print("\non entry rows")
print(ev.loc[F == 1, "tm_status"].value_counts().to_string())
print("\non seeded entries")
print(ev.loc[(F == 1) & (C == 1), "tm_status"].value_counts().to_string())

# what the threshold-free rerun can win back, history exists but the threshold blocks the profile
below = ev["tm_status"].str.startswith("below_threshold").values
cand = below & (ev["n_prior_papers"].values >= 1)
jt_ok = set(map(tuple, ev.loc[ev.tm_status == "ok", ["journal_id", "t"]].drop_duplicates().values))
print(f"\nrows blocked only by the threshold, pre-t history present  {cand.sum():,}")
for lab, m in [("entries", (F == 1) & cand), ("seeded entries", (F == 1) & cand & (C == 1))]:
    hj = pd.Series(map(tuple, ev.loc[m, ["journal_id", "t"]].values)).isin(jt_ok).values
    print(f"  {lab} that gain a T once it falls  {int(hj.sum()):,} of {m.sum():,} candidates")
print("\nceiling after removal, seeded entries about 1,781 of 1,784 covered, entries stay near 12%,")
print("most entrants have no pre-t history and no threshold change can invent one")

T exists on 623,510 rows (9.7%)
  on entry rows            7.3%
  on seeded rows           68.1%
  on seeded entries        63.8%

why the rest is empty, all rows


tm_status
below_threshold_unproductive     5386995
ok                                623510
no_author_history                 286625
no_author_and_journal_history      67643
no_journal_history                 35910
below_threshold_no_abstract        21875

on entry rows
tm_status
below_threshold_unproductive     82907
ok                                7020
no_author_history                 6178
no_author_and_journal_history      454
below_threshold_no_abstract        223
no_journal_history                  37

on seeded entries
tm_status
ok                              1138
below_threshold_unproductive     629
below_threshold_no_abstract       14
no_author_history                  3



rows blocked only by the threshold, pre-t history present  478,374
  entries that gain a T once it falls  4,812 of 4,833 candidates
  seeded entries that gain a T once it falls  643 of 643 candidates

ceiling after removal, seeded entries about 1,781 of 1,784 covered, entries stay near 12%,
most entrants have no pre-t history and no threshold change can invent one


## Step 7: provisional adjustment, F ~ C + T on the rows where T exists

Provisional for three reasons, the threshold timing above, the rolling anchor, a co-author may already have pulled the author's topics toward the journal so part of the effect is adjusted away, and the complete-case population, which is visibly different, its naive ratio alone shows that. Numbers here are for direction and magnitude, not for the report.

- logistic regression, standard errors clustered by author
- the adjusted ratio comes from predicting every row once with `C = 1` and once with `C = 0` and dividing the mean risks, step 7 to 8 of the planning notebook
- overlap is checked first, adjustment only means something where seeded and unseeded rows share the same range of `T`

In [7]:
m = has
X = np.column_stack([np.ones(m.sum()), C[m].astype(float), ev["T"].values[m]])
y = F[m].astype(float)
print(f"complete cases {int(m.sum()):,} rows, {int(y.sum()):,} entries")
print(f"naive ratio inside this population {(y[X[:,1]==1].mean())/(y[X[:,1]==0].mean()):.2f}, "
      f"against 6.32 in the full table, so the population shift is real\n")

# overlap in T between the two exposure groups
q = [0.05, 0.25, 0.5, 0.75, 0.95]
qt1 = np.quantile(X[X[:, 1] == 1, 2], q)
qt0 = np.quantile(X[X[:, 1] == 0, 2], q)
print("T quantiles 5/25/50/75/95")
print("  with seed    " + "  ".join(f"{v:.2f}" for v in qt1))
print("  without      " + "  ".join(f"{v:.2f}" for v in qt0))
print()

beta = np.zeros(3)
for _ in range(25):   # newton, converges in a handful of steps on this size
    p = 1 / (1 + np.exp(-(X @ beta)))
    H = (X * (p * (1 - p))[:, None]).T @ X
    step = np.linalg.solve(H, X.T @ (y - p))
    beta += step
    if np.abs(step).max() < 1e-10:
        break

# sandwich variance with author clusters
p = 1 / (1 + np.exp(-(X @ beta)))
cl = codes[m]
U = X * (y - p)[:, None]
order = np.argsort(cl)
S = np.zeros((3, 3))
for blk in np.split(U[order], np.flatnonzero(np.diff(cl[order])) + 1):
    s = blk.sum(0)
    S += np.outer(s, s)
Hinv = np.linalg.inv((X * (p * (1 - p))[:, None]).T @ X)
se = np.sqrt(np.diag(Hinv @ S @ Hinv))

print(f"C   log odds {beta[1]:.3f}, OR {np.exp(beta[1]):.2f}, "
      f"cluster 95% CI [{np.exp(beta[1]-1.96*se[1]):.2f}, {np.exp(beta[1]+1.96*se[1]):.2f}]")
print(f"T   log odds {beta[2]:.3f}, per 0.1 of topic fit OR {np.exp(beta[2]/10):.2f}")

p1g = (1 / (1 + np.exp(-(np.column_stack([X[:, 0], np.ones(len(X)), X[:, 2]]) @ beta)))).mean()
p0g = (1 / (1 + np.exp(-(np.column_stack([X[:, 0], np.zeros(len(X)), X[:, 2]]) @ beta)))).mean()
print(f"\nadjusted rate ratio, g computation  {p1g/p0g:.2f}")

complete cases 623,510 rows, 7,020 entries
naive ratio inside this population 9.11, against 6.32 in the full table, so the population shift is real

T quantiles 5/25/50/75/95
  with seed    0.75  0.81  0.86  0.90  0.94
  without      0.65  0.73  0.78  0.83  0.89



C   log odds 1.716, OR 5.56, cluster 95% CI [5.15, 6.00]
T   log odds 8.724, per 0.1 of topic fit OR 2.39

adjusted rate ratio, g computation  5.24


## Summary

| number | value | status |
|---|---|---|
| naive rate ratio, full table | 6.32, CI 6.04 to 6.59 | solid as a naive ceiling |
| seeded entries | 1,784, of which 756 rides and 1,028 independent | solid |
| per year | stable around 5.5 to 6.9 from 2021 | solid from 2021 on |
| naive ratio, complete cases | 9.11 | shows the population shift |
| topic-adjusted ratio | 5.24 | provisional, diagnostic run |

Topic fit absorbs real confounding, the ratio drops from 9.11 to 5.24 inside the same population, and the seed effect survives it. The adjusted number stays provisional until the threshold-free export, but the diagnostic run settled the open questions, the timing of T itself is clean, the eligibility threshold is the leak and it goes, and every empty cell now names its reason.

Unblocked next, in order

1. Pierre's threshold-free rerun, also with the real counts in `n_profile_papers` and `n_journal_papers` and the journal profile question settled, then this notebook reruns unchanged and the seeded entries reach about 1,781 of 1,784 with a T
2. the group decision on unknown T, keep T continuous and carry unknown as its own state through `tm_status`, and the frozen variant runs only inside C = 1 as a side check, rolling stays the main model, which already matches `T_pre` in the structure document
3. Lennart's Prolog event table, parity against `../data/event_table_python_v0_oppA.csv`, then the numbers stop being single sourced
4. the full model of the planning notebook, prior papers, career age, journal breadth, journal and year effects, and the planted-effect calibration, once 1 to 3 are settled